# Duka Electronics H1 2026 Executive Board Visual Report
**Author:** Asman Makero Musa · PLP Data Analytics Cohort 7

## Storyboard

- **Audience:** Duka Electronics Board of Directors and Chief Executive Officer (non-technical leadership with 5 minutes to review).
- **Decision Owned:** Allocation of H2 2026 commercial budgets, regional marketing funds, and branch support resources.
- **The Big Idea:** Nairobi drives 36% of revenue, but Q2 momentum has shifted: Eldoret grew 50% on Q1 while Mombasa and Nakuru each fell by about a quarter. H2 budget should prioritise recovering Mombasa and Nakuru.
- **Situation–Complication–Resolution (SCR) Framework:**
  - *Situation:* Duka Electronics generated KES 71.8M across 1,000 orders in H1 2026, anchored by strong Nairobi branch volume and high-ticket laptop sales.
  - *Complication:* Top-line revenue momentum stalled after April's KES 13.5M peak, dropping 19.3% to KES 10.9M in June, driven by severe Q2 contractions in Mombasa (-25%) and Nakuru (-27%).
  - *Resolution:* Reallocate H2 marketing capital toward targeted recovery campaigns in Mombasa and Nakuru, protect Nairobi's core volume, and expand online-to-store bundles.

## Chart Makeover: Before vs. After

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sales = pd.read_csv('duka_sales.csv', parse_dates=['order_date'])
sales['revenue'] = sales['quantity'] * sales['unit_price']

# 1. Before: Unsorted rainbow vertical bar chart
prod_rev = sales.groupby('product')['revenue'].sum()
plt.figure(figsize=(6, 3.5))
plt.bar(prod_rev.index, prod_rev.values, color=['blue', 'orange', 'green', 'red', 'purple'])
plt.title('Revenue by Product'); plt.xlabel('Product'); plt.ylabel('Revenue'); plt.grid(True)
plt.tight_layout(); plt.savefig('figures/before.png', dpi=150); plt.close()

# 2. After: Professional makeover
sorted_prod = prod_rev.sort_values()
top_p = sorted_prod.idxmax()
colors = ['#8E0000' if p == top_p else '#C7C9D1' for p in sorted_prod.index]
fig, ax = plt.subplots(figsize=(6.5, 3.5))
bars = ax.barh(sorted_prod.index, sorted_prod.values / 1e6, color=colors)
for bar, v in zip(bars, sorted_prod.values / 1e6):
    ax.text(v + 0.5, bar.get_y() + bar.get_height() / 2, f'KES {v:.1f}M', va='center', fontsize=9)
share = sorted_prod.max() / sorted_prod.sum() * 100
ax.set_title(f'{top_p} Generates {share:.0f}% of Total Revenue (KES {sorted_prod.max()/1e6:.1f}M)', loc='left', fontweight='bold', fontsize=10)
for s in ['top', 'right', 'bottom']: ax.spines[s].set_visible(False)
ax.set_xticks([]); ax.text(0,-0.15,'Source: Duka POS system, Jan–Jun 2026', transform=ax.transAxes, fontsize=8, color='grey')
plt.tight_layout(); plt.savefig('figures/after.png', dpi=150); plt.close()
print('Generated figures/before.png and figures/after.png')

Generated figures/before.png and figures/after.png


### Makeover Design Rationale

- **Decluttering:** Removed distracting gridlines, top/right/bottom spines, and vertical tick marks to maximize data-ink ratio.
- **Intentional Palette:** Replaced arbitrary rainbow colors with a neutral grey (`#C7C9D1`) base and reserved PLP Redberry (`#8E0000`) for the headline driver (Laptop).
- **Visual Hierarchy:** Switched to horizontal bars sorted from largest to smallest for natural scanning.
- **Insight Title & Source:** Replaced generic 'Revenue by Product' with an explicit conclusion calculated in code, adding direct value labels and source attribution.

## Executive Visual Report: Component Charts & Single-Page Assembly

In [2]:
sales['quarter'] = np.where(sales['order_date'].dt.month <= 3, 'Q1', 'Q2')
monthly = sales.set_index('order_date').resample('MS')['revenue'].sum() / 1e6
branch_tot = (sales.groupby('branch')['revenue'].sum()/1e6).sort_values()
branch_q = pd.pivot_table(sales, index='branch', columns='quarter', values='revenue', aggfunc='sum') / 1e6
channel_rev = (sales.groupby('channel')['revenue'].sum() / 1e6).sort_values()

# Single-Page Visual Report Assembly
fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.8], hspace=0.4, wspace=0.25)

# Panel 1: Trend Line
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(monthly.index.strftime('%b'), monthly.values, color='#8E0000', lw=2.5, marker='o')
pk = monthly.values.argmax()
ax1.annotate(f'Peak: KES {monthly.max():.1f}M', xy=(pk, monthly.max()), xytext=(pk+0.2, monthly.max()-0.8),
             arrowprops=dict(arrowstyle='->'), fontweight='bold',
 fontsize=8.5)
drop = (monthly.iloc[-1] - monthly.max()) / monthly.max() * 100
ax1.set_title(f'1. Revenue Peaked in April (KES {monthly.max():.1f}M), Falling {abs(drop):.0f}% by June', loc='left', fontweight='bold', fontsize=10)
ax1.set_ylabel('Revenue (KES M)'); ax1.spines[['top','right']].set_visible(False); ax1.grid(axis='y', alpha=0.3)
ax1.text(0, -0.22, 'Source: Duka POS system, Jan–Jun 2026', transform=ax1.transAxes, fontsize=7.5, color='grey')

# Panel 2: Branch Totals
ax2 = fig.add_subplot(gs[0, 1])
b_cols = ['#8E0000' if b == branch_tot.idxmax() else '#C7C9D1' for b in branch_tot.index]
b_bars = ax2.barh(branch_tot.index, branch_tot.values, color=b_cols)
for bar, v in zip(b_bars, branch_tot.values): ax2.text(v + 0.3, bar.get_y()+bar.get_height()/2, f'KES {v:.1f}M', va='center', fontsize=8.5)
n_share = branch_tot.max() / branch_tot.sum() * 100
ax2.set_title(f'2. Nairobi Leads Performance with {n_share:.0f}% of Revenue (KES {branch_tot.max():.1f}M)', loc='left', fontweight='bold', fontsize=10)
ax2.spines[['top','right','bottom']].set_visible(False); ax2.set_xticks([])
ax2.text(0, -0.22, 'Source: Duka POS system, Jan–Jun 2026', transform=ax2.transAxes, fontsize=7.5, color='grey')

# Panel 3: Momentum Shift (Q1 vs Q2)
ax3 = fig.add_subplot(gs[1, 0])
x = np.arange(len(branch_q)); w = 0.35
ax3.bar(x - w/2, branch_q['Q1'], width=w, label='Q1', color='#C7C9D1')
q2_cols = ['#8E0000' if b in ['Mombasa', 'Nakuru'] else '#0BA184' for b in branch_q.index]
ax3.bar(x + w/2, branch_q['Q2'], width=w, label='Q2', color=q2_cols)
ax3.set_xticks(x); ax3.set_xticklabels(branch_q.index); ax3.set_ylabel('Revenue (KES M)')
ax3.legend(frameon=False, loc='upper left'); ax3.spines[['top','right']].set_visible(False); ax3.grid(axis='y', alpha=0.3)
ax3.set_title('3. Q2 Shifts: Eldoret Rose +50%, while Mombasa (-25%) & Nakuru (-27%) Contracted', loc='left', fontweight='bold', fontsize=10)
ax3.text(0, -0.22, 'Source: Duka POS system, Jan–Jun 2026', transform=ax3.transAxes, fontsize=7.5, color='grey')

# Panel 4: Products
ax4 = fig.add_subplot(gs[1, 1])
p_rev = (sales.groupby('product')['revenue'].sum()/1e6).sort_values()
p_cols = ['#8E0000' if p=='Laptop' else ('#0BA184' if p=='Tablet' else '#C7C9D1') for p in p_rev.index]
p_bars = ax4.barh(p_rev.index, p_rev.values, color=p_cols)
for bar, v in zip(p_bars, p_rev.values): ax4.text(v + 0.3, bar.get_y()+bar.get_height()/2, f'KES {v:.1f}M', va='center', fontsize=8.5)
lap_share = p_rev['Laptop'] / p_rev.sum() * 100
ax4.set_title(f'4. Laptops (KES {p_rev["Laptop"]:.1f}M) & Tablets Drive {(p_rev["Laptop"]+p_rev["Tablet"])/p_rev.sum()*100:.0f}% of Revenue', loc='left', fontweight='bold', fontsize=10)
ax4.spines[['top','right','bottom']].set_visible(False); ax4.set_xticks([])
ax4.text(0, -0.22, 'Source: Duka POS system, Jan–Jun 2026', transform=ax4.transAxes, fontsize=7.5, color='grey')

# Panel 5: Channels
ax5 = fig.add_subplot(gs[2, :])
c_cols = ['#C7C9D1', '#0BA184', '#8E0000']
c_bars = ax5.barh(channel_rev.index, channel_rev.values, color=c_cols,
 height=0.45)
for bar, v in zip(c_bars, channel_rev.values):
    ax5.text(v + 0.4, bar.get_y()+bar.get_height()/2, f'KES {v:.1f}M ({v/channel_rev.sum()*100:.1f}%)', va='center', fontsize=8.5)
ax5.set_title('5. In-Store Channel Dominates (55.4%); Online Channel Contributes KES 25.0M (34.9%)', loc='left', fontweight='bold', fontsize=10)
ax5.spines[['top','right','bottom']].set_visible(False); ax5.set_xticks([])
ax5.text(0, -0.22, 'Source: Duka POS system, Jan–Jun 2026', transform=ax5.transAxes, fontsize=7.5, color='grey')

fig.suptitle('Duka Electronics H1 2026 Board Visual Report: Core Stability & Regional Allocation', fontsize=12.5, fontweight='bold', y=0.98)
plt.savefig('reports/duka_board_report.png', dpi=200, bbox_inches='tight')
plt.savefig('reports/duka_board_report.pdf', bbox_inches='tight')
plt.close()
print('Saved reports/duka_board_report.png and reports/duka_board_report.pdf')

Saved reports/duka_board_report.png and reports/duka_board_report.pdf


### Board Executive Summary & Speaker Notes (150 Words)

Duka Electronics achieved total revenue of KES 71.8M across 1,000 orders in H1 2026, anchored by strong commercial operations in Nairobi (KES 26.1M, 36.4% share) and high-margin laptop sales (KES 41.5M, 57.9% share). However, company momentum decelerated significantly post-April, dropping 19.3% to KES 10.9M in June due to steep second-quarter branch contractions in Mombasa (-25%) and Nakuru (-27%), despite Eldoret surging +50% to KES 5.3M. In-store retail remains the primary commercial engine at 55.4%, complemented by online sales at 34.9%.

We recommend reallocating H2 regional marketing expenditure toward recovering footfall in Mombasa and Nakuru, maintaining inventory scaffolding in fast-growing Eldoret, and expanding high-value online-to-store hardware promotional bundles. To optimize regional margins, branch operations must streamline supply chain turnaround times across western regions and establish weekly sales performance checkpoints. Note that historical point-of-sale transaction records capture completed customer purchases only, excluding marketing promotional costs, operating overhead, and external customer product returns.